In [8]:
from pathlib import Path

import h5py
import numpy as np

from md_Helpers import cavitation


# ============================================================
# Select and load/create the cavitation initial state
# ============================================================

cavitation_result = cavitation.get_or_create_cavitation_state(
    n_fcc_cells=30,
    target_rho=0.755,
    kT=0.800,
    source_nsteps=1_000_000,
    radius=2.500,

    source_seed=1,
    source_phase_name="randomization",

    random_location=False,
    bubble_center=None,
    bubble_seed=1,

    overwrite=False,
    overwrite_source=False,

    # Do not launch a missing thermalization run.
    create_source_if_missing=False,
)


# ============================================================
# Print the returned result
# ============================================================

print("\nCAVITATION RESULT")
print("=" * 100)
print("Status:              ", cavitation_result.get("status"))
print("Created new:         ", cavitation_result.get("created_new"))

print("\nPATHS")
for name, value in cavitation_result.get("paths", {}).items():
    print(f"{name:25} {value}")

print("\nCREATION INFO")
creation_info = cavitation_result.get("creation_info", {})

if creation_info:
    for name in sorted(creation_info):
        value = creation_info[name]

        if isinstance(value, np.ndarray):
            print(
                f"{name:25} ndarray, "
                f"shape={value.shape}, dtype={value.dtype}"
            )
        else:
            print(f"{name:25} {value}")
else:
    print("(no creation information returned)")

print("\nSOURCE RESULT")
source_result = cavitation_result.get("source_result", {})
print("Created new:         ", source_result.get("created_new"))

for name, value in source_result.get("paths", {}).items():
    print(f"source_{name:18} {value}")


# ============================================================
# Locate the cavitation creation-metadata file
# ============================================================

metadata_path = Path(
    cavitation_result["paths"]["creation_metadata_path"]
)

print("\nCAVITATION METADATA FILE")
print("=" * 100)
print(metadata_path)

if not metadata_path.exists():
    raise FileNotFoundError(
        "No cavitation metadata file exists. "
        f"Cavitation status was: {cavitation_result.get('status')!r}\n"
        f"Expected file: {metadata_path}"
    )


# ============================================================
# Print every /metadata/... section
# ============================================================

def readable(value):
    """Convert HDF5 and NumPy values into readable values."""
    if isinstance(value, bytes):
        return value.decode("utf-8", errors="replace")

    if isinstance(value, np.generic):
        return value.item()

    if isinstance(value, np.ndarray):
        return np.array2string(
            value,
            threshold=np.inf,
            max_line_width=120,
        )

    return value


with h5py.File(metadata_path, "r") as hdf:
    if "metadata" not in hdf:
        raise KeyError(f"No /metadata group exists in {metadata_path}")

    metadata_groups = ["/metadata"]

    def find_groups(name, obj):
        if isinstance(obj, h5py.Group):
            metadata_groups.append(f"/metadata/{name}")

    hdf["metadata"].visititems(find_groups)

    for group_path in metadata_groups:
        group = hdf[group_path]

        print("\n")
        print("=" * 100)
        print(group_path)
        print("=" * 100)

        item_number = 1

        # Attributes saved directly on this group
        if group.attrs:
            print("\nATTRIBUTES")

            for name in sorted(group.attrs):
                value = readable(group.attrs[name])

                print(f"\n[{item_number}] {name}")
                print("    storage: attribute")
                print(f"    type:    {type(value).__name__}")
                print(f"    value:   {value}")

                item_number += 1

        # Datasets saved directly inside this group
        datasets = [
            (name, obj)
            for name, obj in group.items()
            if isinstance(obj, h5py.Dataset)
        ]

        if datasets:
            print("\nDATASETS")

            for name, dataset in sorted(datasets):
                value = readable(dataset[()])

                print(f"\n[{item_number}] {name}")
                print("    storage: dataset")
                print(f"    dtype:   {dataset.dtype}")
                print(f"    shape:   {dataset.shape}")
                print(f"    value:\n{value}")

                item_number += 1

        if not group.attrs and not datasets:
            print("\n(empty container group)")

Thermalized state exists: checking phase separation.
Loaded existing thermalized state:
/exp/e961/data/MDsims-data/pnichols/Thermalized_States_v3/FCC/n_cells_30/rho_0.755/kT_0.800/nsteps_1000000/seed_1/randomization.gsd
Thermalized state good; continuing to cavitation.


Loaded existing cavitation initial state:
/exp/e961/data/MDsims-data/pnichols/Cavitation_States_v3/FCC/n_cells_30/source_rho_0.755/kT_0.800/source_nsteps_1000000/source_seed_1/source_phase_randomization/radius_2.500/center_box/cavitation_initial.gsd

CAVITATION RESULT
Status:               loaded_initial
Created new:          False

PATHS
folder                    /exp/e961/data/MDsims-data/pnichols/Cavitation_States_v3/FCC/n_cells_30/source_rho_0.755/kT_0.800/source_nsteps_1000000/source_seed_1/source_phase_randomization/radius_2.500/center_box
state_path                /exp/e961/data/MDsims-data/pnichols/Cavitation_States_v3/FCC/n_cells_30/source_rho_0.755/kT_0.800/source_nsteps_1000000/source_seed_1/source_phase_ran

In [13]:
# ============================================================
# Clean metadata in all existing cavitation initial states
# ============================================================

from importlib import reload

from md_Helpers import metadata as metadata_helpers
from md_Helpers.paths import CAVITATION_STATES_V3_ROOT

# Reload so the notebook sees the updated helper code.
reload(metadata_helpers)

# First run with False to preview.
# After reviewing the output, change to True and rerun.
APPLY_CHANGES = False

cleanup_result = (
    metadata_helpers.run_cavitation_creation_metadata_cleanup(
        root=CAVITATION_STATES_V3_ROOT,
        apply_changes=APPLY_CHANGES,
        show_changed_files=False,
    )
)

CAVITATION INITIAL-STATE METADATA CLEANUP
Root:           /exp/e961/data/MDsims-data/pnichols/Cavitation_States_v3
Mode:           DRY RUN
Files checked:  729

STATUS
already_clean   729

FIELDS
No fields need cleanup.

ERRORS
No errors found.

DRY RUN ONLY: no files were modified.
Set APPLY_CHANGES = True and rerun to perform the cleanup.
